# SIPARTA: RANCANG BANGUN ALAT DETEKSI DAN PERINGATAN DINI GAS BERACUN DARI REAKSI KIMIA RUMAH TANGGA BERBASIS JARINGAN SARAF TIRUAN

Notebook ini merepresentasikan **seluruh evolusi riset SIPARTA** untuk kompetisi PKM-KC, dari validasi teori kimia hingga implementasi akhir pada Edge-AI (Raspberry Pi).

Alur penelitian dibagi menjadi tiga fase utama:
1. **Fase 1: Proof of Concept (Reaksi Kimia)** - Memvalidasi teori 10 parameter (pH, konsentrasi, dll) untuk memprediksi tingkat bahaya pencampuran kimia.
2. **Fase 2: Translasi Sensor Fisik (IoT)** - Mereduksi 10 parameter teoretis menjadi 4 parameter bacaan sensor gas aktual untuk implementasi hardware (Edge-AI).
3. **Fase 3: Simulasi IoT & Inference** - Menguji keandalan sistem klasifikasi dalam *loop* operasional perangkat keras.



In [305]:
RUN_MODE = 'TRAINING'
import os
import time
import json
import pickle
import logging
import requests
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

print(f"TensorFlow Version: {tf.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")

# Reprodusibilitas
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)


TensorFlow Version: 2.21.0
NumPy Version: 2.5.3
Pandas Version: 3.0.6


---
# FASE 1: PROOF OF CONCEPT (Teori Reaksi Kimia)
Pada fase ini, model dilatih dengan **10 fitur teoretis** yang merepresentasikan parameter kimia (pH, viskositas, konsentrasi larutan, dll). Tujuannya adalah membuktikan bahwa kombinasi kimiawi dapat diklasifikasikan risikonya menjadi AMAN, WASPADA, dan BAHAYA.


In [307]:
# Load base dataset untuk keperluan EDA agar sel-sel di bawah tidak Error
try:
    df = pd.read_csv('siparta_sensor_dataset.csv')
    X = df[['mics5524', 'tgs2600', 'mq2', 'mq135']].values
    y = df['label'].values
    NUM_CLASSES = 3
except Exception:
    pass


In [309]:
print('[INFO] Cell ini diabaikan karena dataset kini langsung dikelola lewat siparta_sensor_dataset.csv')


[INFO] Cell ini diabaikan karena dataset kini langsung dikelola lewat siparta_sensor_dataset.csv


In [310]:
# ============================================================
# Informasi Dataset
# ============================================================
print("=" * 60)
print("INFORMASI DATASET SIPARTA")
print("=" * 60)
print(f"Jumlah sampel  : {df.shape[0]}")
print(f"Jumlah fitur   : {df.shape[1]}")
print(f"Missing values : {df.isnull().sum().sum()}")
print(f"\nTipe data:")
print(df.dtypes)
print(f"\nStatistik deskriptif:")
df.describe()


INFORMASI DATASET SIPARTA
Jumlah sampel  : 1
Jumlah fitur   : 6
Missing values : 0

Tipe data:
mics5524    float64
tgs2600     float64
mq2         float64
mq135       float64
label         int64
source          str
dtype: object

Statistik deskriptif:


,mics5524,tgs2600,mq2,mq135,label
count,1.0,1.0,1.00,1.00,1.0
mean,0.1,0.2,0.15,0.05,0.0
std,NaN,NaN,NaN,NaN,NaN
min,0.1,0.2,0.15,0.05,0.0
25%,0.1,0.2,0.15,0.05,0.0
50%,0.1,0.2,0.15,0.05,0.0
75%,0.1,0.2,0.15,0.05,0.0
max,0.1,0.2,0.15,0.05,0.0


In [311]:
# ============================================================
# Distribusi Tingkat Risiko
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if df.empty:
    axes[0].text(0.5, 0.5, 'No actual data available', ha='center', va='center', fontsize=12)
    axes[1].text(0.5, 0.5, 'No actual data available', ha='center', va='center', fontsize=12)
    for ax in axes: ax.axis('off')
else:
    # Bar plot (pastikan 3 kelas selalu terwakili meski data kosong untuk kelas tersebut)
    risk_counts = df['label'].value_counts().reindex([0, 1, 2], fill_value=0)
    risk_labels = ['Aman (0)', 'Sedang (1)', 'Tinggi (2)']
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    
    bars = axes[0].bar(risk_labels, risk_counts.values, color=colors, edgecolor='black', linewidth=0.8)
    axes[0].set_title('Distribusi Tingkat Risiko Gas Beracun', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Jumlah Sampel')
    
    for bar, val in zip(bars, risk_counts.values):
        axes[0].text(bar.get_x() + bar.get_width()/2, val + (0.5 if val == 0 else val*0.05), str(val),
                    ha='center', fontweight='bold', fontsize=12)
                    
    # Pie chart
    if risk_counts.sum() > 0:
        # Hanya plot pie untuk kelas yang memiliki nilai > 0
        pie_counts = risk_counts[risk_counts > 0]
        pie_labels = [risk_labels[i] for i in pie_counts.index]
        pie_colors = [colors[i] for i in pie_counts.index]
        axes[1].pie(pie_counts.values, labels=pie_labels, autopct='%1.1f%%',
                    colors=pie_colors, startangle=90, textprops={'fontsize': 11})
    else:
        axes[1].text(0.5, 0.5, 'No data for pie chart', ha='center')
        axes[1].axis('off')
        
    axes[1].set_title('Proporsi Tingkat Risiko', fontsize=14, fontweight='bold')
    
plt.tight_layout()
plt.show()



[SKIP] Visualisasi dilewati karena jumlah data terlalu sedikit (hanya data tes).


In [314]:
# ============================================================
# Split data: Train (70%), Validation (15%), Test (15%)
# ============================================================
import numpy as np

# Select only numeric features, drop target 'label' and non-feature strings
X = df.drop(columns=['label', 'label_name', 'timestamp'], errors='ignore')
# if there are other non-numeric columns, drop them as well
X = X.select_dtypes(include=[np.number])
y = df['label']

if len(X) < 5:
    print('[SKIP] Dataset Phase 1 sangat kecil. Mem-bypass train_test_split.')
    X_p1_train = X_p1_val = X_p1_test = X_temp = X
    y_p1_train = y_p1_val = y_p1_test = y_temp = y
else:
    X_p1_train, X_temp, y_p1_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=SEED, stratify=y
    )
    X_p1_val, X_p1_test, y_p1_val, y_p1_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
    )

print(f"Jumlah Set Pelatihan : {X_p1_train.shape[0]}")
print(f"Jumlah Set Validasi  : {X_p1_val.shape[0]}")
print(f"Jumlah Set Pengujian : {X_p1_test.shape[0]}")



[SKIP] Dataset sangat kecil. Mem-bypass train_test_split (hanya untuk testing)


In [315]:
# ============================================================
# Feature Scaling (StandardScaler)
# ============================================================
if len(X_p1_train) > 0:
    scaler_p1 = StandardScaler()
    X_p1_train_scaled = scaler_p1.fit_transform(X_p1_train)
    X_p1_val_scaled = scaler_p1.transform(X_p1_val)
    X_p1_test_scaled = scaler_p1.transform(X_p1_test)
    print("[SUCCESS] Feature scaling selesai (StandardScaler)")
    print(f"   Mean training (sesudah): {X_p1_train_scaled.mean():.6f}")
    print(f"   Std training (sesudah) : {X_p1_train_scaled.std():.6f}")
else:
    print("[SKIP] Tidak ada data untuk di-scale.")
    X_p1_train_scaled = X_p1_val_scaled = X_p1_test_scaled = X_p1_train



[SUCCESS] Feature scaling selesai (StandardScaler)
   Mean training (sesudah): 0.000000
   Std training (sesudah) : 0.000000


In [316]:
# ============================================================
# One-Hot Encoding target (untuk multi-class classification)
# ============================================================
y_p1_train_cat = to_categorical(y_p1_train, num_classes=NUM_CLASSES)
y_p1_val_cat = to_categorical(y_p1_val, num_classes=NUM_CLASSES)
y_p1_test_cat = to_categorical(y_p1_test, num_classes=NUM_CLASSES)

print(f"[SUCCESS] One-hot encoding selesai")
print(f"   y_p1_train shape: {y_p1_train_cat.shape}")
print(f"   Contoh: {y_p1_train[:5]} -> ")
print(f"           {y_p1_train_cat[:5]}")


[SUCCESS] One-hot encoding selesai
   y_p1_train shape: (1, 3)
   Contoh: [0] -> 
           [[1. 0. 0.]]


In [317]:
# ============================================================
# Hyperparameter
# ============================================================
INPUT_DIM = X_p1_train_scaled.shape[1]
LEARNING_RATE = 0.001
EPOCHS = 200
BATCH_SIZE = 32

print(f"Input dimension : {INPUT_DIM}")
print(f"Output classes  : {NUM_CLASSES}")
print(f"Learning rate   : {LEARNING_RATE}")
print(f"Epochs          : {EPOCHS}")
print(f"Batch size      : {BATCH_SIZE}")


Input dimension : 4
Output classes  : 3
Learning rate   : 0.001
Epochs          : 200
Batch size      : 32


In [318]:
# ============================================================
# Bangun Model JST
# ============================================================
def build_ann_model(input_dim, num_classes, learning_rate=0.001):
    """
    Membangun arsitektur JST untuk klasifikasi tingkat risiko
    gas beracun dari reaksi kimia rumah tangga.

    Args:
        input_dim: Jumlah fitur input
        num_classes: Jumlah kelas output (3: Aman/Sedang/Tinggi)
        learning_rate: Learning rate untuk optimizer Adam

    Returns:
        model: Compiled Keras Sequential model
    """
    model_p1 = Sequential(name='SIPARTA_JST')

    # Input + Hidden Layer 1
    model_p1.add(Input(shape=(input_dim,)))
    model_p1.add(Dense(128, activation='relu', kernel_initializer='he_normal', name='hidden_1'))
    model_p1.add(BatchNormalization(name='bn_1'))
    model_p1.add(Dropout(0.3, name='dropout_1'))

    # Hidden Layer 2
    model_p1.add(Dense(64, activation='relu', kernel_initializer='he_normal', name='hidden_2'))
    model_p1.add(BatchNormalization(name='bn_2'))
    model_p1.add(Dropout(0.3, name='dropout_2'))

    # Hidden Layer 3
    model_p1.add(Dense(32, activation='relu', kernel_initializer='he_normal', name='hidden_3'))
    model_p1.add(BatchNormalization(name='bn_3'))
    model_p1.add(Dropout(0.2, name='dropout_3'))

    # Output Layer (3 kelas: Aman, Sedang, Tinggi)
    model_p1.add(Dense(num_classes, activation='softmax', name='output'))

    # Compile
    optimizer = Adam(learning_rate=learning_rate)
    model_p1.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model_p1


# Buat model
model_p1 = build_ann_model(INPUT_DIM, NUM_CLASSES, LEARNING_RATE)

# Tampilkan arsitektur
model_p1.summary()


Model: "SIPARTA_JST"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_1 (Dense)                │ (None, 128)            │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_1 (BatchNormalization)       │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_2 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_2 (BatchNormalization)       │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_3 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_3 (BatchNormalization)       │ (None, 32)             │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,971 (46.76 KB)

 Trainable params: 11,523 (45.01 KB)

 Non-trainable params: 448 (1.75 KB)

In [319]:
# ============================================================
# Definisi Callbacks
# ============================================================

# Early Stopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

# Reduce Learning Rate
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=10,
    min_lr=1e-7,
    verbose=1
)

# Model Checkpoint
checkpoint = ModelCheckpoint(
    'siparta_best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='max',
    verbose=1
)

callbacks_list = [early_stopping, reduce_lr, checkpoint]
print("[SUCCESS] Callbacks dikonfigurasi: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint")


[SUCCESS] Callbacks dikonfigurasi: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


In [320]:
# ============================================================
# Training Model (Phase 1)
# ============================================================
if len(X_p1_train_scaled) > 0:
    history_p1 = model_p1.fit(
        X_p1_train_scaled, y_p1_train_cat,
        epochs=30,
        batch_size=32,
        validation_data=(X_p1_val_scaled, y_p1_val_cat),
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )
    print("[SUCCESS] Training Model Fase 1 Selesai.")
else:
    print("[SKIP] Data kosong, tidak dapat melatih model.")
    history_p1 = None




[START] Memulai training model SIPARTA JST...

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 1.0000 - loss: 1.0986
Epoch 1: val_loss improved from None to 1.09203, saving model to siparta_best_model.keras

Epoch 1: finished saving model to siparta_best_model.keras
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 1.0986 - val_accuracy: 1.0000 - val_loss: 1.0920 - learning_rate: 0.0010
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 1.0918
Epoch 2: val_loss did not improve from 1.09203
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 1.0000 - loss: 1.0918 - val_accuracy: 1.0000 - val_loss: 1.0857 - learning_rate: 0.0010
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 1.0857
Epoch 3: val_loss did not improve from 1.09203
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 1.0000 - loss: 1.0857 - val_accuracy: 1.0000 - val_loss: 1.0792 - learning_rate: 0.0010
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/st

In [321]:
# ============================================================
# Evaluasi pada Test Set
# ============================================================
if len(X_p1_test_scaled) > 0:
    test_loss, test_accuracy = model_p1.evaluate(X_p1_test_scaled, y_p1_test_cat, verbose=0)
    print("=" * 55)
    print("  HASIL EVALUASI MODEL SIPARTA JST - TEST SET")
    print("=" * 55)
    print(f"  Test Loss     : {test_loss:.4f}")
    print(f"  Test Accuracy : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    print("=" * 55)
else:
    print("[SKIP] Data uji kosong, evaluasi model dilewati.")



  HASIL EVALUASI MODEL SIPARTA JST - TEST SET
  Test Loss     : 0.0760
  Test Accuracy : 1.0000 (100.00%)


In [322]:
if len(X) < 5:
    print('[SKIP] Dataset terlalu kecil untuk dievaluasi secara statistik (classification_report/confusion_matrix).')
else:
    # ============================================================
    # Prediksi & Classification Report
    # ============================================================
    y_pred_proba = model_p1.predict(X_p1_test_scaled, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    print("=" * 55)
    print("CLASSIFICATION REPORT")
    print("=" * 55)
    target_names = ['Aman (0)', 'Sedang (1)', 'Tinggi (2)']
    print(classification_report(y_p1_test, y_pred, target_names=target_names))



[SKIP] Dataset terlalu kecil untuk dievaluasi secara statistik (classification_report/confusion_matrix).


In [323]:
if len(X) < 5:
    print('[SKIP] Dataset terlalu kecil untuk dievaluasi secara statistik (classification_report/confusion_matrix).')
else:
    # ============================================================
    # Confusion Matrix
    # ============================================================
    cm = confusion_matrix(y_p1_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
                xticklabels=target_names,
                yticklabels=target_names,
                linewidths=1, linecolor='black')
    plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
    plt.ylabel('Actual (Label Asli)', fontsize=12)
    plt.xlabel('Predicted (Prediksi Model)', fontsize=12)
    plt.tight_layout()
    plt.show()
    # Hitung per-kelas accuracy
    print("\nAkurasi per kelas:")
    for i, name in enumerate(target_names):
        if cm[i].sum() > 0:
            acc = cm[i][i] / cm[i].sum()
            print(f"  {name}: {acc:.4f} ({cm[i][i]}/{cm[i].sum()})")



[SKIP] Dataset terlalu kecil untuk dievaluasi secara statistik (classification_report/confusion_matrix).


---
# FASE 2: TRANSLASI SENSOR FISIK & PELATIHAN EDGE AI (IoT)
Model dengan 10 fitur kimia di atas tidak dapat digunakan secara langsung pada alat fisik karena sensor yang digunakan (MQ-2, MQ-135, TGS2600, MiCS-5524) mengukur *voltage* analog dari partikel gas, bukan konsentrasi larutan kimia.
Oleh karena itu, kita mereduksi *input shape* menjadi 4 dimensi, dan menggunakan arsitektur *Dense Network* yang lebih ringan (Edge-AI) agar sanggup berjalan secara kontinu (real-time) di dalam memori Raspberry Pi tanpa menyebabkan kelebihan beban termal.

> **[FORENSIC AUDIT MARKER - DATASET SIMULASI SENSOR]**
> Untuk memvalidasi arsitektur jaringan sensor 4-dimensi, digunakan data sintetis berbatas (*bounded random*) yang menyerupai karakter bacaan ADC. Data ini tidak diklaim sebagai data empiris IoT aktual.


In [324]:
# KONFIGURASI MODE EKSEKUSI
# Pilih salah satu: 'TRAINING', 'SIMULATION', 'HARDWARE_TEST', 'PRODUCTION_INTEGRATION'
RUN_MODE = 'TRAINING'

# Konfigurasi Simulasi/Loop
MAX_SIMULATION_ITERATIONS = 5

print(f"SIPARTA Notebook Mode: {RUN_MODE}")


SIPARTA Notebook Mode: TRAINING


In [325]:
import os
import pandas as pd
import numpy as np

def load_actual_sensor_data():
    dataset_path = "siparta_sensor_dataset.csv"
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"[DATASET_NOT_FOUND] File {dataset_path} tidak ditemukan.")
    
    df = pd.read_csv(dataset_path)
    
    # Validasi kolom wajib
    required_cols = ['mics5524', 'tgs2600', 'mq2', 'mq135', 'label', 'source']
    for col in required_cols:
        if col not in df.columns:
            raise KeyError(f"[ABORT] Kolom '{col}' wajib ada dalam dataset sensor.")
            
    # Filter dummy
    df = df[df['source'] == 'iot'].copy()
    iot_df = df
    
    print("=== DATASET AUDIT ===")
    print(f"Total IoT records   : {len(df)}")
    print()
    
    return df, iot_df

if RUN_MODE == 'TRAINING':
    df, iot_df = load_actual_sensor_data()
    if df.empty:
        print("[WARNING] Tidak ada data IoT aktual di siparta_sensor_dataset.csv")
    else:
        display(df.head())



=== DATASET AUDIT ===
Total records       : 1
Actual IoT records  : 0
Dummy test records  : 1

[WARNING] TEST MODE: Pipeline execution allowed with 1 dummy record.
[WARNING] Metrik evaluasi yang dihasilkan TIDAK VALID secara nyata karena dataset belum memadai.


,mics5524,tgs2600,mq2,mq135,label,source
0,0.1,0.2,0.15,0.05,0,dummy_test


In [326]:
if RUN_MODE == 'TRAINING':
    print("=" * 60)
    print("INFORMASI DATASET SENSOR SIPARTA")
    print("=" * 60)
    print(df.info())
    print("\nStatistik Deskriptif:")
    display(df.describe())
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    if df.empty:
        axes[0].text(0.5, 0.5, 'No actual data available', ha='center', va='center')
        axes[1].text(0.5, 0.5, 'No actual data available', ha='center', va='center')
        for ax in axes: ax.axis('off')
    else:
        # Distribusi Label
        sns.countplot(data=df, x='label', ax=axes[0], order=[0, 1, 2], palette=['#2ecc71', '#f39c12', '#e74c3c'])
        axes[0].set_title("Distribusi Label Risiko")
        
        # Korelasi antar sensor
        if len(df) > 1 and df[['mics5524', 'tgs2600', 'mq2', 'mq135']].var().sum() > 0:
            sns.heatmap(df[['mics5524', 'tgs2600', 'mq2', 'mq135']].corr(), annot=True, cmap='coolwarm', ax=axes[1])
        else:
            axes[1].text(0.5, 0.5, 'Korelasi tidak terdefinisi\n(data terlalu sedikit/konstan)', ha='center', va='center')
            axes[1].axis('off')
        axes[1].set_title("Korelasi Fitur Sensor")
        
    plt.tight_layout()
    plt.show()



[SKIP] Visualisasi dilewati karena jumlah data terlalu sedikit (hanya data tes).


In [327]:
# ============================================================
# Preprocessing Phase 2: Split & Scaling
# ============================================================
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

X = df[['mics5524', 'tgs2600', 'mq2', 'mq135']].values
y = df['label'].values

if len(X) < 5:
    print('[SKIP] Dataset Phase 2 sangat kecil. Mem-bypass train_test_split.')
    X_train_scaled = X_test_scaled = X
    y_train_cat = y_test_cat = y
    y_test = y
    scaler = StandardScaler()
    if len(X) > 0:
        scaler.fit(X)
else:
    # Phase 2 split: 80% Train, 20% Test (atau 70/30)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    
    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # One Hot Encoding
    y_train_cat = to_categorical(y_train, num_classes=3)
    y_test_cat = to_categorical(y_test, num_classes=3)
    
    print(f"Jumlah Set Pelatihan (Phase 2): {X_train.shape[0]}")
    print(f"Jumlah Set Pengujian (Phase 2): {X_test.shape[0]}")



[SKIP] Dataset sangat kecil. Mem-bypass train_test_split (hanya untuk testing)


In [328]:
def build_edge_model(input_dim=4, num_classes=3):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation='relu', input_shape=(input_dim,), name='hidden_1'),
        tf.keras.layers.Dropout(0.2, name='dropout_1'),
        tf.keras.layers.Dense(8, activation='relu', name='hidden_2'),
        tf.keras.layers.Dense(num_classes, activation='softmax', name='output')
    ], name='SIPARTA_JST')
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

if RUN_MODE == 'TRAINING':
    model = build_edge_model()
    model.summary()
    
    # Training Model
    if len(X_train_scaled) > 0:
        history = model.fit(
            X_train_scaled, y_train_cat, 
            epochs=15, 
            batch_size=32, 
            validation_split=(0.2 if len(X_train_scaled) >= 5 else 0.0), 
            verbose=1
        )
    else:
        print("[SKIP] Data IoT kosong, tidak dapat melatih model (Phase 2).")
        history = None



/home/brayone-xv/Documents/My File/CODE/lomba/pkm-kc/SIPARTA/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "SIPARTA_JST"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_1 (Dense)                │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_2 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243 (972.00 B)

 Trainable params: 243 (972.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 832ms/step - accuracy: 1.0000 - loss: 1.0797
Epoch 2/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 1.0000 - loss: 1.0593
Epoch 3/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 1.0000 - loss: 1.0162
Epoch 4/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 1.0000 - loss: 0.9714
Epoch 5/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 1.0000 - loss: 0.9307
Epoch 6/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 1.0000 - loss: 0.8591
Epoch 7/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 1.0000 - loss: 0.8397
Epoch 8/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 1.0000 - loss: 0.8065
Epoch 9/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 1.0000 - loss: 0.7884
Epoch 10/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 1.0000 - loss: 0.7037
Epoch 11/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 1.0000 - loss: 0.6921
Epoch 12/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 1.0000 - loss: 0.6185


In [329]:
if RUN_MODE == 'TRAINING':
    if len(df) == 0:
        print('[SKIP] Dataset kosong, tidak ada evaluasi.')
    else:
        # Evaluasi Akurasi
        loss, acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
        print(f"\nTest Accuracy: {acc*100:.2f}%")
        
        # Plot History
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        if 'loss' in history.history:
            axes[0].plot(history.history['loss'], label='Train Loss')
        if 'val_loss' in history.history:
            axes[0].plot(history.history['val_loss'], label='Val Loss')
        axes[0].set_title('Model Loss')
        axes[0].legend()
        
        if 'accuracy' in history.history:
            axes[1].plot(history.history['accuracy'], label='Train Acc')
        if 'val_accuracy' in history.history:
            axes[1].plot(history.history['val_accuracy'], label='Val Acc')
        axes[1].set_title('Model Accuracy')
        axes[1].legend()
        plt.show()
        
        # Classification Report
        y_pred_proba = model.predict(X_test_scaled, verbose=0)
        y_pred = np.argmax(y_pred_proba, axis=-1)
        target_names = ["AMAN", "WASPADA", "BAHAYA"]
        
        print("\nClassification Report:")
        try:
            from sklearn.metrics import classification_report, confusion_matrix
            print(classification_report(y_test, y_pred, target_names=target_names, labels=[0, 1, 2], zero_division=0))
        except Exception as e:
            print(f"[SKIP] Classification report gagal: {e}")
            
        # Confusion Matrix
        try:
            cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
            plt.figure(figsize=(6, 4))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
            plt.title("Confusion Matrix")
            plt.xlabel("Predicted")
            plt.ylabel("Actual")
            plt.show()
        except Exception as e:
            print(f"[SKIP] Confusion matrix gagal: {e}")



[SKIP] Dataset terlalu kecil untuk dievaluasi secara statistik (classification_report/confusion_matrix).


In [330]:
if RUN_MODE == 'TRAINING':
    import os
    import pickle
    os.makedirs("model", exist_ok=True)
    
    if len(X_train_scaled) > 0:
        # 1. Simpan Scaler
        with open("model/siparta_scaler.pkl", "wb") as f:
            pickle.dump(scaler, f)
            
        # 2. Simpan Keras Model
        model.save("model/siparta_ann.h5")
        
        # 3. Konversi ke TensorFlow Lite
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        tflite_model = converter.convert()
        with open('model/siparta_model.tflite', 'wb') as f:
            f.write(tflite_model)
        
        print("[SUCCESS] Model ANN, TFLite, dan Scaler berhasil disimpan di folder 'model/'.")
    else:
        print("[SKIP] Penyimpanan model dilewati karena tidak ada data pelatihan.")



INFO:tensorflow:Assets written to: /tmp/tmp_jk7bikz/assets


INFO:tensorflow:Assets written to: /tmp/tmp_jk7bikz/assets


Saved artifact at '/tmp/tmp_jk7bikz'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 4), dtype=tf.float32, name='keras_tensor_600')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  123775929433168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123775929422800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123775929423568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123775929420304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123775929420112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123775929430480: TensorSpec(shape=(), dtype=tf.resource, name=None)
[SUCCESS] Seluruh artefak model (Scaler, Keras, TFLite, Config) berhasil disimpan ke 'model/'.
[SUCCESS] Artefak model & dataset berhasil disalin ke ../web_backend/fastapi/models untuk deployment backend.


W0000 00:00:1790943007.263045  504601 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1790943007.263081  504601 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1790943007.263258  504601 reader.cc:83] Reading SavedModel from: /tmp/tmp_jk7bikz
I0000 00:00:1790943007.263736  504601 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1790943007.263751  504601 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp_jk7bikz
I0000 00:00:1790943007.267213  504601 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1790943007.286742  504601 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp_jk7bikz
I0000 00:00:1790943007.292419  504601 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 29169 microseconds.


---
# FASE 3: SIMULASI INFERENCE & IOT LOOP
Pada tahap akhir ini, kita mendemonstrasikan bagaimana model yang telah dilatih (TFLite) di-*deploy* dalam *environment* perangkat keras. 
Script di bawah ini bertindak sebagai *Inference Engine* untuk klasifikasi langsung dan pengiriman *payload* ke backend FastAPI.


In [331]:
class InferenceEngine:
    def __init__(self, model_path="model/siparta_ann.tflite", scaler_path="model/siparta_scaler.pkl"):
        self.use_fallback = False
        self.classes = ["AMAN", "WASPADA", "BAHAYA"]
        try:
            self.interpreter = tf.lite.Interpreter(model_path=model_path)
            self.interpreter.allocate_tensors()
            self.input_details = self.interpreter.get_input_details()
            self.output_details = self.interpreter.get_output_details()
            
            with open(scaler_path, 'rb') as f:
                self.scaler = pickle.load(f)
            print("[SUCCESS] InferenceEngine: TFLite Model dan Scaler berhasil diload.")
        except Exception as e:
            print(f"[WARNING] Gagal meload model/scaler: {e}")
            self.use_fallback = True

    def preprocess(self, sensor_values):
        # Clipping ekstrim (Fail-safe)
        clamped = [max(0.0, min(5.0, v)) for v in sensor_values]
        input_data = np.array([clamped], dtype=np.float32)
        if hasattr(self, 'scaler'):
            return self.scaler.transform(input_data).astype(np.float32)
        return input_data

    def predict(self, sensor_values):
        if self.use_fallback:
            return "ERROR_MODEL_NOT_LOADED"
            
        try:
            input_data = self.preprocess(sensor_values)
            self.interpreter.set_tensor(self.input_details[0]['index'], input_data)
            self.interpreter.invoke()
            output = self.interpreter.get_tensor(self.output_details[0]['index'])[0]
            
            pred_class = np.argmax(output)
            confidence = output[pred_class]
            return self.classes[pred_class], confidence
        except Exception as e:
            print(f"Inference error: {e}")
            return "ERROR_INFERENCE_FAILED", 0.0

# Instantiate engine (digunakan saat eksekusi loop IoT)
engine = InferenceEngine()


[SUCCESS] InferenceEngine: TFLite Model dan Scaler berhasil diload.


/home/brayone-xv/Documents/My File/CODE/lomba/pkm-kc/SIPARTA/.venv/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [332]:
print("\n=== Memulai IoT Loop Mode: [PRODUCTION_INTEGRATION] ===")
print("[FAIL] Dummy data dihapus. Loop ini membutuhkan RPi fisik yang berjalan lewat main_rpi.py")




=== Memulai IoT Loop Mode: [PRODUCTION_INTEGRATION] ===
[FAIL] Dummy data dihapus. Loop ini membutuhkan RPi fisik yang berjalan lewat main_rpi.py
